# Spheroid morphology: a first pass at automated image analysis

**Abiha Baig — first attempt, July 2026**

I am a wet-lab cancer biologist. I have not done automated image analysis before.
This notebook is me working out, from scratch, how you get from a microscopy image
to the four morphology features named in the UvA/Amsterdam UMC spheroid project:

- spheroid size
- cell cluster count
- area distribution
- inter-cluster distance

I start with **classical thresholding**, not deep learning, because I wanted to
understand what segmentation actually *is* before reaching for a model. As it
turns out, watching the classical method fail is the clearest possible explanation
of why the field needs learned models at all.

Everything here runs in the browser. Nothing to install.


---
## 0. How to run this

Top menu: **Runtime → Run all**. Or click each cell and press **Shift+Enter**.

A cell is running when it shows `[*]`. It's done when it shows a number.


## 1. Setup

`import` means "load a toolbox someone else wrote."
- **numpy** — arrays of numbers. An image *is* an array of numbers.
- **scikit-image** — image analysis.
- **matplotlib** — plots.
- **pandas** — tables.


In [ ]:
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from skimage import filters, measure, morphology, segmentation, color
from scipy import ndimage as ndi
from scipy.spatial.distance import pdist, squareform

print("Ready.")


## 2. Make a test image where I know the right answer

Before trusting a method on real data, I want to test it on data where I already
know the truth — the same logic as running a positive control.

So: I place **6 spheroids** at positions I choose myself. Two of them (bottom right)
I deliberately place touching, because that is what happens in a real well.

A microscope image is just a grid of numbers — brightness at each pixel. Below I
build that grid directly: for each spheroid, pixels near its centre get a high
number, pixels far away get zero. Then I add random noise, because real cameras
are noisy.


In [ ]:
rng = np.random.default_rng(0)

img = np.zeros((512, 512), dtype=float)
yy, xx = np.mgrid[0:512, 0:512]

# (row, column, radius) - I am choosing these. This is my ground truth.
true_spheroids = [(90,100,28), (120,300,35), (300,120,42),
                  (330,360,30), (360,400,26), (200,220,20)]

for cy, cx, r in true_spheroids:
    dist = np.sqrt((yy-cy)**2 + (xx-cx)**2)
    img += np.clip(1.0 - (dist/r)**2, 0, 1) * 200

img += rng.normal(0, 8, img.shape)      # camera noise
img = np.clip(img, 0, 255)

print("Image shape:", img.shape)
print("I placed", len(true_spheroids), "spheroids. Remember this number.")


In [ ]:
plt.figure(figsize=(6,6))
plt.imshow(img, cmap='gray')
plt.title("Synthetic spheroid field - 6 spheroids, two touching")
plt.axis('off')
plt.show()


## 3. Segmentation, one step at a time

**Segmentation** = deciding which pixels are object and which are background.
That's all it is.

### 3a. Blur slightly

Noise makes individual pixels jump around. A small blur averages each pixel with
its neighbours so the boundary decision isn't made on a single noisy pixel.
`sigma=2` is the blur radius. Try changing it.


In [ ]:
smoothed = filters.gaussian(img, sigma=2)

fig, ax = plt.subplots(1, 2, figsize=(11,5))
ax[0].imshow(img, cmap='gray');      ax[0].set_title("raw");      ax[0].axis('off')
ax[1].imshow(smoothed, cmap='gray'); ax[1].set_title("blurred");  ax[1].axis('off')
plt.show()


### 3b. Choose a brightness cutoff (Otsu's method)

Now I need a number: above it = spheroid, below it = background.

Rather than guess, **Otsu's method** picks the cutoff automatically. It tries every
possible cutoff and keeps the one that best separates the pixels into two groups.
The histogram below shows why it works — the pixels form two humps (dark
background, bright spheroid) and Otsu finds the valley between them.


In [ ]:
threshold = filters.threshold_otsu(smoothed)
print("Otsu chose a cutoff of:", round(threshold, 1))

plt.figure(figsize=(7,3.5))
plt.hist(smoothed.ravel(), bins=100, color='steelblue')
plt.axvline(threshold, color='red', lw=2, label=f'Otsu cutoff = {threshold:.1f}')
plt.yscale('log')
plt.xlabel("pixel brightness"); plt.ylabel("count (log)")
plt.legend(); plt.title("Where to cut?")
plt.show()


In [ ]:
binary = smoothed > threshold          # True/False for every pixel
binary = ndi.binary_fill_holes(binary)  # fill any gaps inside objects
binary = morphology.remove_small_objects(binary, min_size=150)  # drop debris

plt.figure(figsize=(6,6))
plt.imshow(binary, cmap='gray')
plt.title("Binary mask: white = spheroid, black = background")
plt.axis('off')
plt.show()


### 3c. Label the separate objects

`measure.label` walks the mask and gives every connected white blob its own number.
**This is where it goes wrong.**


In [ ]:
labels = measure.label(binary)
n_found = labels.max()

print("I placed:", len(true_spheroids), "spheroids")
print("The method found:", n_found)
print()
if n_found != len(true_spheroids):
    print(">>> IT IS WRONG. Off by", len(true_spheroids) - n_found)

plt.figure(figsize=(6,6))
plt.imshow(color.label2rgb(labels, bg_label=0))
plt.title(f"Each colour = one detected object ({n_found} found)")
plt.axis('off')
plt.show()


### 3d. Why it failed — and why this is the important cell

Look at the bottom right. The two touching spheroids came out **one colour**. They
are connected in the mask, so `label` cannot know they are two things.

This is *the* central failure of threshold-based segmentation, and it is not a bug
I can tune away. Thresholding only knows brightness. It has no concept of "a
spheroid is a roughly round thing" — so it cannot know that a peanut shape is
really two circles.

**Learned models like Cellpose and StarDist exist precisely to fix this.** They have
been trained on thousands of annotated images, so they carry a learned notion of
what a cell or spheroid *is shaped like*, and they will split touching objects that
thresholding fuses.

That is what "AI-driven image segmentation" in the job ad is actually buying you.

I can even detect the failure without knowing ground truth — the merged object
gives itself away by its shape.


In [ ]:
props = measure.regionprops_table(labels, intensity_image=img,
    properties=('label','area','equivalent_diameter_area',
                'centroid','eccentricity','solidity'))
df = pd.DataFrame(props)
df['suspect_merge'] = (df.solidity < 0.9) | (df.eccentricity > 0.7)

print(df.round(2).to_string(index=False))
print()
print("Flagged as probable merges:", int(df.suspect_merge.sum()))


**Read that table.** Every genuine spheroid: `eccentricity` near 0 (round),
`solidity` near 0.98 (no dents). The merged object: eccentricity **0.91** (stretched),
solidity **0.83** (waisted). The shape statistics betray it.

So a crude but real quality check falls out for free: flag any object that isn't
round enough to be a single spheroid.


## 4. The four features the project asks for


In [ ]:
centroids = df[['centroid-0','centroid-1']].values
distances = squareform(pdist(centroids))
np.fill_diagonal(distances, np.inf)
nearest = distances.min(axis=1)

print("1. SPHEROID SIZE (equivalent diameter, px)")
print("   mean", round(df.equivalent_diameter_area.mean(),1),
      "| range", round(df.equivalent_diameter_area.min(),1),
      "-", round(df.equivalent_diameter_area.max(),1))
print()
print("2. CLUSTER COUNT:", len(df), "  <-- known to be an undercount")
print()
print("3. AREA DISTRIBUTION (px^2)")
print("   mean", round(df.area.mean(),1), "| sd", round(df.area.std(),1),
      "| CV", round(100*df.area.std()/df.area.mean(),1), "%")
print()
print("4. INTER-CLUSTER DISTANCE (px)")
print("   mean nearest-neighbour", round(nearest.mean(),1))
print("   mean all-pairs", round(pdist(centroids).mean(),1))


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14,4))

ax[0].hist(df.area, bins=8, color='steelblue', edgecolor='k')
ax[0].set_xlabel('area (px$^2$)'); ax[0].set_ylabel('count')
ax[0].set_title('Area distribution')

ax[1].scatter(df.equivalent_diameter_area, df.solidity,
              c=np.where(df.suspect_merge,'crimson','steelblue'), s=70)
ax[1].axhline(0.9, ls='--', c='grey')
ax[1].set_xlabel('diameter (px)'); ax[1].set_ylabel('solidity')
ax[1].set_title('Red = flagged as merged')

ax[2].hist(nearest, bins=6, color='seagreen', edgecolor='k')
ax[2].set_xlabel('nearest-neighbour distance (px)')
ax[2].set_title('Inter-cluster distance')

plt.tight_layout(); plt.show()


## 5. A classical fix: watershed

Before jumping to deep learning, there is a classical trick worth knowing, because
an interviewer may well ask "did you try watershed?"

Idea: for each mask pixel, compute distance to the nearest background pixel. The
centre of each spheroid is furthest from background, so each spheroid has a peak.
Treat those peaks as seeds and let them "flood" outward until they meet. The
meeting line becomes the split.


In [ ]:
distance_map = ndi.distance_transform_edt(binary)
coords = morphology.local_maxima(filters.gaussian(distance_map, 3))
markers = measure.label(coords)
ws = segmentation.watershed(-distance_map, markers, mask=binary)

print("Threshold alone found:", labels.max())
print("Threshold + watershed found:", ws.max())
print("Truth:", len(true_spheroids))

fig, ax = plt.subplots(1, 2, figsize=(11,5))
ax[0].imshow(color.label2rgb(labels, bg_label=0)); ax[0].set_title(f"threshold ({labels.max()})"); ax[0].axis('off')
ax[1].imshow(color.label2rgb(ws, bg_label=0));     ax[1].set_title(f"+ watershed ({ws.max()})");   ax[1].axis('off')
plt.show()


Watershed can recover the split — but it is fragile. It depends on the blur
`sigma` I chose for peak-finding; too little and one spheroid gets split in two,
too much and the peaks merge. **Try changing the `3` in `filters.gaussian(distance_map, 3)`
and watch the count break.** That brittleness — a hand-tuned parameter that has to
be re-tuned per dataset — is the other reason the field moved to learned models.


## 6. What I would do next

1. **Real images.** Swap the synthetic field for public spheroid/organoid microscopy
   (e.g. the Broad Bioimage Benchmark Collection). Everything below the image-loading
   cell stays the same.
2. **Cellpose.** Run a pretrained model on the same images and compare its count
   against threshold and watershed. The comparison is the result.
3. **Beyond 2D.** These are single planes. Real spheroids are 3D objects; area
   becomes volume and the geometry gets harder.
4. **Toward the model.** These features — size, count, area spread, spacing — are
   exactly the state variables a cellular Potts model would need to be calibrated
   against. Getting them out of an image reliably is the first link in that chain.

## What I actually learned

- Segmentation is a decision about pixels, and thresholding makes that decision
  using brightness alone.
- Objects that touch get fused, and no amount of tuning fixes it, because the method
  has no notion of shape.
- Shape statistics (solidity, eccentricity) can flag those failures automatically.
- Watershed patches it, at the cost of a parameter that needs re-tuning per dataset.
- That is the gap learned models fill — and now I understand what they are for
  rather than just that they are used.
